# VisionBridge — Real Video Base Model Validation

Run this notebook from top to bottom. It lets you upload/select **one real ISL video**, extracts the exact Pose/Face landmark contract used by VisionBridge, loads the committed `base_model.pt`, and prints **Ground Truth vs Predicted**, confidence, blank ratio, CER, and diagnostic information.

This notebook is for validation only. It does **not** train or modify the model.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil

REPO = Path('/content/VisionBridge')
if not REPO.exists():
    !git clone https://github.com/BharathWaj-K-R/VisionBridge.git /content/VisionBridge
else:
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'], check=True)

print('Repository:', REPO)
print('HEAD:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'], text=True).strip())

In [ ]:
# Select/upload the real ISL video.
from google.colab import files

print('Upload ONE real ISL sentence video (mp4/mov/avi).')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No video was uploaded.')

VIDEO_SRC = Path('/content') / next(iter(uploaded))
if VIDEO_SRC.suffix.lower() not in {'.mp4','.mov','.avi','.mkv'}:
    raise ValueError(f'Unsupported video format: {VIDEO_SRC.suffix}')

VIDEO_DIR = REPO / 'data' / 'model_check' / 'video'
VIDEO_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_PATH = VIDEO_DIR / VIDEO_SRC.name
shutil.copy2(VIDEO_SRC, VIDEO_PATH)

print('Selected video:', VIDEO_PATH)

In [ ]:
# Optional ground-truth text. Enter the exact English sentence corresponding to the selected video.
GROUND_TRUTH = input('Ground truth English sentence (leave blank if unknown): ').strip()
print('Ground truth:', repr(GROUND_TRUTH) if GROUND_TRUTH else '(not supplied)')

In [ ]:
# Create an isolated Python 3.12 environment for legacy MediaPipe Holistic.
# This avoids the Colab Python/MediaPipe compatibility problems seen earlier.
import os, subprocess, textwrap

UV = '/usr/local/bin/uv'
if not Path(UV).exists():
    subprocess.run(['bash','-lc','curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
    UV = '/root/.local/bin/uv'

MP_ENV = Path('/content/visionbridge_mp312')
if not MP_ENV.exists():
    subprocess.run([UV,'python','install','3.12'], check=True)
    subprocess.run([UV,'venv','--python','3.12',str(MP_ENV)], check=True)

mp_python = str(MP_ENV / 'bin' / 'python')
subprocess.run([UV,'pip','install','--python',mp_python,'mediapipe==0.10.21','numpy==1.26.4','opencv-python-headless','pandas','matplotlib'], check=True)
print('MediaPipe environment ready:', mp_python)

In [ ]:
# Extract real Pose [T,132] and Face [T,1404] using the repository's own extractor.
# A temporary one-row CSV is used because the extractor expects a UID + text manifest.
import csv

CHECK_DIR = REPO / 'data' / 'model_check'
PROCESSED = CHECK_DIR / 'processed'
VIDEOS = CHECK_DIR / 'videos'
VIDEOS.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

UID = VIDEO_PATH.stem
EXTRACT_VIDEO = VIDEOS / f'{UID}.mp4'
shutil.copy2(VIDEO_PATH, EXTRACT_VIDEO)
LABELS = CHECK_DIR / 'validation_labels.csv'
with LABELS.open('w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['uid','text'])
    w.writerow([UID, GROUND_TRUTH or 'validation sample'])

env = os.environ.copy()
env['MPLBACKEND'] = 'Agg'
cmd = [mp_python, str(REPO/'backend'/'scripts'/'extract_keypoints.py'), '--videos_dir', str(VIDEOS), '--labels_csv', str(LABELS), '--out_dir', str(PROCESSED)]
result = subprocess.run(cmd, cwd=str(REPO), env=env, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('MediaPipe extraction failed.')

import numpy as np
pose_path = PROCESSED/'pose'/f'{UID}.npy'
face_path = PROCESSED/'face'/f'{UID}.npy'
pose = np.load(pose_path)
face = np.load(face_path)
print('Pose shape:', pose.shape)
print('Face shape:', face.shape)
if pose.ndim != 2 or pose.shape[1] != 132: raise ValueError(f'Invalid pose shape: {pose.shape}')
if face.ndim != 2 or face.shape[1] != 1404: raise ValueError(f'Invalid face shape: {face.shape}')
if pose.shape[0] != face.shape[0]: raise ValueError('Pose/face frame counts differ.')
print('REAL KEYPOINT EXTRACTION: PASS')

In [ ]:
# Load the exact committed base model and run inference on the selected video.
sys.path.insert(0, str(REPO/'backend'))
import torch
from app.models.base_model import load_frozen_base_model
from app.training.isltranslate import SimpleCharTokenizer, _downsample_to_max_length

WEIGHTS = REPO/'backend'/'app'/'models'/'weights'/'base_model.pt'
VOCAB = REPO/'backend'/'app'/'models'/'weights'/'base_model.vocab.json'
if not WEIGHTS.exists(): raise FileNotFoundError(WEIGHTS)
if not VOCAB.exists(): raise FileNotFoundError(VOCAB)

tokenizer = SimpleCharTokenizer.load(VOCAB)
model = load_frozen_base_model(str(WEIGHTS), vocab_size=tokenizer.vocab_size)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device).eval()

pose_t, face_t = _downsample_to_max_length(torch.from_numpy(pose).float(), torch.from_numpy(face).float(), UID)
pose_t = pose_t.unsqueeze(0).to(device)
face_t = face_t.unsqueeze(0).to(device)
lengths = torch.tensor([pose_t.shape[1]], dtype=torch.long, device=device)

with torch.no_grad():
    logits = model(pose_t, face_t, lengths=lengths)

print('Device:', device)
print('Logits shape:', tuple(logits.shape))
print('Logits finite:', bool(torch.isfinite(logits).all()))

In [ ]:
# Deterministic greedy CTC decoding + validation report.
# Uses the same blank-id=0 contract as the training tokenizer.
import math

logit = logits[0, :int(lengths[0].item())]
log_probs = torch.log_softmax(logit, dim=-1)
probs = log_probs.exp()
frame_ids = probs.argmax(dim=-1)
frame_conf = probs.max(dim=-1).values

# CTC collapse: remove consecutive duplicates, then remove blank (0).
collapsed = []
previous = None
for token_id in frame_ids.tolist():
    if token_id != previous:
        collapsed.append(token_id)
    previous = token_id
decoded_ids = [i for i in collapsed if i != 0]

def ids_to_text(ids):
    chars = []
    for i in ids:
        if 0 <= i < len(tokenizer.id_to_token):
            token = tokenizer.id_to_token[i]
            if token != tokenizer.blank_token:
                chars.append(token)
    return ''.join(chars).strip()

prediction = ids_to_text(decoded_ids) or '(no sign detected)'
blank_count = int((frame_ids == 0).sum().item())
non_blank = int((frame_ids != 0).sum().item())
blank_ratio = blank_count / max(1, frame_ids.numel())
confidence = float(frame_conf.mean().item())

def levenshtein(a, b):
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [0]*len(b)
        for j, cb in enumerate(b, 1):
            cur[j] = min(prev[j]+1, cur[j-1]+1, prev[j-1]+(ca != cb))
        prev = cur
    return prev[-1]

cer = None
if GROUND_TRUTH:
    cer = levenshtein(prediction.lower(), GROUND_TRUTH.lower()) / max(1, len(GROUND_TRUTH))

print('\n' + '='*70)
print('VISIONBRIDGE REAL-VIDEO MODEL CHECK')
print('='*70)
print('VIDEO:          ', VIDEO_PATH.name)
print('DEVICE:         ', device)
print('FRAMES:         ', frame_ids.numel())
print('GROUND TRUTH:   ', GROUND_TRUTH or '(not supplied)')
print('PREDICTED:      ', prediction)
print('CONFIDENCE:     ', f'{confidence:.3f}')
print('BLANK RATIO:    ', f'{blank_ratio:.4f}')
print('NON-BLANK:      ', non_blank)
print('UNIQUE TOKENS:  ', len(set(decoded_ids)))
print('LOGITS FINITE:  ', bool(torch.isfinite(logits).all()))
if cer is not None: print('CER:             ', f'{cer:.4f}')
print('='*70)

if blank_ratio == 1.0:
    print('RESULT: MODEL PRODUCED ONLY CTC BLANKS ON THIS VIDEO.')
elif prediction == '(no sign detected)':
    print('RESULT: NON-BLANK FRAMES EXIST, BUT CTC COLLAPSED TO EMPTY TEXT.')
else:
    print('RESULT: NON-BLANK TEXT PREDICTION PRODUCED.')

In [ ]:
# Optional: show the uploaded video for visual confirmation.
from IPython.display import Video, display
display(Video(str(VIDEO_PATH), embed=True))